# YOLO11s + Shape-IoU on Kaggle

这份 notebook 只改回归损失，不改模型结构。

实验目标：用 Shape-IoU 替换原版 YOLO11s 的 CIoU，观察 `drill_pipe` 的定位误差是否下降。

In [ ]:
# Cell 1: 拉代码
%cd /kaggle/working
!rm -rf yolov11s_kuangjing
!git clone https://github.com/tuanziya666/yolov11s_kuangjing.git
%cd /kaggle/working/yolov11s_kuangjing
!git checkout main
!git pull

In [ ]:
# Cell 2: 检查 Shape-IoU 相关文件
!ls -lh train_yolo11s_shape_iou.py
!ls -lh ultralytics/utils/loss.py
!python -c "import sys; sys.path.insert(0, '/kaggle/working/yolov11s_kuangjing'); from ultralytics import YOLO; print('repo ok')"

In [ ]:
# Cell 3: 自动定位 Kaggle 数据集根目录
from pathlib import Path

CANDIDATES = [
    Path('/kaggle/input/yolo-kuangjing-mixed25k-dataset'),
    Path('/kaggle/input/datasets/yuanfangshang/yolo-kuangjing-mixed25k-dataset'),
    Path('/kaggle/input'),
]

def find_dataset_root(candidates):
    for base in candidates:
        if not base.exists():
            continue
        if (base / 'images' / 'train').exists() and (base / 'labels' / 'train').exists():
            return base
        for p in base.rglob('*'):
            if p.is_dir() and (p / 'images' / 'train').exists() and (p / 'labels' / 'train').exists():
                return p
    return None

DATASET_ROOT = find_dataset_root(CANDIDATES)
if DATASET_ROOT is None:
    raise FileNotFoundError('没找到包含 images/train 和 labels/train 的 Kaggle 数据集根目录，请检查数据集是否已挂载。')

print('Resolved DATASET_ROOT =', DATASET_ROOT)

In [ ]:
# Cell 4: 写 Kaggle 数据集 yaml
from pathlib import Path

yaml_text = f'''path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: chuck
  1: gripper
  2: drill_pipe
  3: coal_miner
'''

yaml_path = Path('/kaggle/working/yolov11s_kuangjing/ultralytics/cfg/datasets/coalmine4_kaggle.yaml')
yaml_path.write_text(yaml_text, encoding='utf-8')
print(yaml_text)

In [ ]:
# Cell 5: 训练参数
RUN_NAME = 'yolo11s_shape_iou_kaggle'
EPOCHS = 100
BATCH = 16
WORKERS = 2
IMGSZ = 640
SEED = 42
SHAPE_SCALE = 0.5

print('RUN_NAME =', RUN_NAME)
print('SHAPE_SCALE =', SHAPE_SCALE)

In [ ]:
# Cell 6: 开始训练
import subprocess
import sys
from pathlib import Path

repo = Path('/kaggle/working/yolov11s_kuangjing')
cmd = [
    sys.executable,
    '-u',
    'train_yolo11s_shape_iou.py',
    '--data', '/kaggle/working/yolov11s_kuangjing/ultralytics/cfg/datasets/coalmine4_kaggle.yaml',
    '--name', RUN_NAME,
    '--project', '/kaggle/working/runs',
    '--device', '0',
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--workers', str(WORKERS),
    '--imgsz', str(IMGSZ),
    '--seed', str(SEED),
    '--shape-scale', str(SHAPE_SCALE),
]

print(' '.join(cmd))
proc = subprocess.Popen(
    cmd,
    cwd=repo,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='')

ret = proc.wait()
print(f'\nProcess return code: {ret}')
if ret != 0:
    raise RuntimeError(f'Training failed with return code {ret}')

In [ ]:
# Cell 7: 用 best.pt 重新评估 val / test
import sys
from pathlib import Path

sys.path.insert(0, '/kaggle/working/yolov11s_kuangjing')
from ultralytics import YOLO

best_pt = Path(f'/kaggle/working/runs/{RUN_NAME}/weights/best.pt')
print('best.pt =', best_pt)

model = YOLO(str(best_pt))

model.val(
    data='/kaggle/working/yolov11s_kuangjing/ultralytics/cfg/datasets/coalmine4_kaggle.yaml',
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    project='/kaggle/working/evals',
    name=f'{RUN_NAME}_val',
)

model.val(
    data='/kaggle/working/yolov11s_kuangjing/ultralytics/cfg/datasets/coalmine4_kaggle.yaml',
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    project='/kaggle/working/evals',
    name=f'{RUN_NAME}_test',
)

In [ ]:
# Cell 8: 打包结果，方便下载
%cd /kaggle/working
!zip -qr {RUN_NAME}_results.zip runs/{RUN_NAME} evals/{RUN_NAME}_val evals/{RUN_NAME}_test
!ls -lh {RUN_NAME}_results.zip